<a href="https://colab.research.google.com/github/rachingham/skills-github-pages/blob/main/Assessment2_pynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import copy
import math
import numpy as np
import cv2
import matplotlib.pyplot as plt
import scipy
import scipy.optimize
import torch
import torchvision
import torchvision.transforms.functional as tvtf

In [38]:
def preprocess_image(image):
    image = tvtf.to_tensor(image)
    image = image.unsqueeze(dim=0)
    return image

# For multiple class_index
def get_detections_subset(maskrcnn, vid, frame_numbers, score_threshold=0.5, class_index=[1]):

    # Record how long the video is (in frames)
    vid_length = int(vid.get(cv2.CAP_PROP_FRAME_COUNT))

    # Read the frame on which the video was previously
    initial_frame = int(vid.get(cv2.CAP_PROP_POS_FRAMES))

    # Set the video the frame 0
    vid.set(cv2.CAP_PROP_POS_FRAMES, 0)

    # For each frame, read it, give it to maskrcnn and record the boxes
    det = {}

    for i, frame_num in enumerate(frame_numbers):
           vid.set(cv2.CAP_PROP_POS_FRAMES,frame_num)
           _, img = vid.read()
           img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

           with torch.no_grad():
              result = maskrcnn(preprocess_image(img))[0]
        # Only check for mult class
              mask1 = torch.zeros(len(result["labels"]), dtype=torch.bool)
              for cls_id in class_index:  # ← CHANGED: Check all class indices
                mask1 |= (result["labels"] == cls_id)
        # And only if the model is confident enough about it
              mask2 = result["scores"] > score_threshold
              mask = mask1 & mask2

              boxes = result["boxes"][mask].detach().cpu().numpy()
              labels = result["labels"][mask].detach().cpu().numpy()
              detections_with_class = np.column_stack([labels, boxes])
              det[frame_num]=detections_with_class

              print(f'{i+1:0d}/{vid_length}')
              print(f'Frame {frame_num}: Found {len(boxes)} detections ({i+1}/{len(frame_numbers)})')
    # Set the video back to the frame it was before we started
    vid.set(cv2.CAP_PROP_POS_FRAMES, initial_frame)

    return det

COLOURS = [
    tuple(int(colour_hex.strip('#')[i:i+2], 16) for i in (0, 2, 4))
    for colour_hex in plt.rcParams['axes.prop_cycle'].by_key()['color']
]


COCO_CLASSES = {
    0: '__background__',
    1: 'person',
    47: 'cup',
    48: 'fork',
    49: 'knife',      # Fixed: was 59 in your comment
    50: 'spoon',
    51: 'bowl',
    52: 'banana',
    77: 'scissors',
}

In [ ]:
def get_detections_subset(maskrcnn, vid, frame_numbers, score_threshold=0.5, class_index=[1]):

    # Record how long the video is (in frames)
    vid_length = int(vid.get(cv2.CAP_PROP_FRAME_COUNT))

    # Read the frame on which the video was previously
    initial_frame = int(vid.get(cv2.CAP_PROP_POS_FRAMES))

    # Set the video the frame 0
    vid.set(cv2.CAP_PROP_POS_FRAMES, 0)

    # For each frame, read it, give it to maskrcnn and record the boxes
    det = {}

    for i, frame_num in enumerate(frame_numbers):
           vid.set(cv2.CAP_PROP_POS_FRAMES,frame_num)
           _, img = vid.read()
           img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

           with torch.no_grad():
              result = maskrcnn(preprocess_image(img))[0]
        # Only check for mult class
              mask1 = torch.zeros(len(result["labels"]), dtype=torch.bool)
              for cls_id in class_index:  # ← CHANGED: Check all class indices
                mask1 |= (result["labels"] == cls_id)
        # And only if the model is confident enough about it
              mask2 = result["scores"] > score_threshold
              mask = mask1 & mask2

              boxes = result["boxes"][mask].detach().cpu().numpy()
              labels = result["labels"][mask].detach().cpu().numpy()
              detections_with_class = np.column_stack([labels, boxes])
              det[frame_num]=detections_with_class

              print(f'{i+1:0d}/{vid_length}')
              print(f'Frame {frame_num}: Found {len(boxes)} detections ({i+1}/{len(frame_numbers)})')
    # Set the video back to the frame it was before we started
    vid.set(cv2.CAP_PROP_POS_FRAMES, initial_frame)

    return det

maskrcnn = torchvision.models.detection.maskrcnn_resnet50_fpn(pretrained=True)
maskrcnn.eval()
vid = cv2.VideoCapture('/content/10092025_162422.MP4')
total_frames = int(vid.get(cv2.CAP_PROP_FRAME_COUNT))

# Every 10th frame
frame_numbers = [int(k) for k in range(0, total_frames, 10)]

# all_detections = get_detections(maskrcnn, vid, class_index=[1,3,4,5,7,8], score_threshold=0.3)
all_detections = get_detections_subset(maskrcnn, vid, frame_numbers, score_threshold=0.3, class_index=[1, 47, 48, 49, 50, 51, 52, 77])
torch.save(all_detections, './assessment2_1.th')

def draw_detections(img, det, class_index, colours=COLOURS):
    for i, ( tlx, tly, brx, bry) in enumerate(det):
        cl=class_index[i]
        i %= len(colours)
        cv2.rectangle(img, (tlx, tly), (brx, bry), color=colours[i], thickness=2)

# Load detections
# The traceback indicates an UnpicklingError related to numpy._core.multiarray._reconstruct.
# Allowing this global should fix the issue.
#torch.serialization.add_safe_globals([np._core.multiarray._reconstruct])
#all_detections = torch.load('./assessment2_1.th')
print (det[0])

vid = cv2.VideoCapture('/content/10092025_162422.MP4')

# NEW_HEIGHT = 600
# scale = NEW_HEIGHT/park_img.shape[0]
def annotate_class(img, det, class_index,class_map, conf=None, colours=COLOURS, ):
    for i, (tlx, tly, brx, bry) in enumerate(det):
        cl=class_index[i]
        # txt = class_map[cl]
        txt = class_map.get(cl, f"Class {cl}")
        if conf is not None:
           txt += f' {conf[i]:1.3f}'
        # A box with a border thickness draws half of that thickness to the left of the
        # boundaries, while filling fills only within the boundaries, so we expand the filled
        # region to match the border
        offset = 1

        cv2.rectangle(img,
                      (tlx-offset, tly-offset-12),
                      (tlx-offset+len(txt)*12, tly),
                      color=colours[i],
                      thickness=cv2.FILLED)

        ff = cv2.FONT_HERSHEY_PLAIN
        cv2.putText(img, txt, (tlx, tly-1), fontFace=ff, fontScale=1.0, color=(255,)*3)
n_rows = len(frame_numbers)
fig, axes = plt.subplots(n_rows, 1, figsize=(22, 13*n_rows))

for idx, fr_num in enumerate(frame_numbers):
    # TODO: Set vid to frame i
    # .set(): Method to set video properties
    # cv2.CAP_PROP_POS_FRAMES: Property flag indicating "frame position"
    # fr_num: The target frame number (e.g., 0, 10, 20, 30)
    vid.set(cv2.CAP_PROP_POS_FRAMES, fr_num)

    # TODO: Read a frame
    _, img = vid.read()
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # TODO: Get detections for frame i out of all_detections
    #det = all_detections[fr_num]

    # TODO: Change det to the np.int32 dtype
    #det = det.astype(np.int32)
    det_full = all_detections[fr_num].astype(np.int32)
    # TODO: Draw on img
    class_indices = det_full[:, 0]  # First column
    bboxes = det_full[:, 1:5]         # Columns 1-4


    # DEBUG: Check the values
    print(f"Frame {fr_num}:")
    print(f"  Class indices: {class_indices}")
    print(f"  Max class index: {class_indices.max()}")
    print(f"  COCO_CLASSES length: {len(COCO_CLASSES)}")




    draw_detections(img, bboxes, class_index=class_indices)
    annotate_class(img, bboxes, class_index=class_indices, class_map=COCO_CLASSES)


    axes[idx].imshow(img)
    axes[idx].axis('off')
    axes[idx].set_title(f'Frame #{fr_num}- {len(bboxes)}objects')
plt.savefig('./detections.png', bbox_inches='tight')
plt.show()
print("saved to './detections.png' ")

print([det_full])
print([det])
print([bboxes])
print("Sample detection shape:", det_full [frame_numbers[0]].shape)
print("Sample detection:", all_detections_with_class[frame_numbers[0]][0])

In [43]:
maskrcnn = torchvision.models.detection.maskrcnn_resnet50_fpn(pretrained=True)
maskrcnn.eval()
vid = cv2.VideoCapture('/content/10092025_162422.MP4')
total_frames = int(vid.get(cv2.CAP_PROP_FRAME_COUNT))

# Every 10th frame
frame_numbers = [int(k) for k in range(0, total_frames, 10)]

# all_detections = get_detections(maskrcnn, vid, class_index=[1,3,4,5,7,8], score_threshold=0.3)
all_detections = get_detections_subset(maskrcnn, vid, frame_numbers, score_threshold=0.3, class_index=[1, 47, 48, 49, 50, 51, 52, 77])
torch.save(all_detections, './assessment2_1.th')

def draw_detections(img, det, colours=COLOURS):
    for i, (tlx, tly, brx, bry) in enumerate(det):
        i %= len(colours)
        cv2.rectangle(img, (tlx, tly), (brx, bry), color=colours[i], thickness=2)

def annotate_class(img, det, class_map, conf=None, colours=COLOURS):
    for i, (class_map, cl, tlx, tly, brx, bry) in enumerate(det):
        txt = Class_map[cl]
        if conf is not None:
            txt += f' {conf[i]:1.3f}'
        # A box with a border thickness draws half of that thickness to the left of the
        # boundaries, while filling fills only within the boundaries, so we expand the filled
        # region to match the border
        offset = 1

        cv2.rectangle(img,
                      (tlx-offset, tly-offset-12),
                      (tlx-offset+len(txt)*12, tly),
                      color=colours[i],
                      thickness=cv2.FILLED)

        ff = cv2.FONT_HERSHEY_PLAIN
        cv2.putText(img, txt, (tlx, tly-1), fontFace=ff, fontScale=1.0, color=(255,)*3)

vid = cv2.VideoCapture('/content/10092025_162422.MP4')

n_rows = len(frame_numbers)
fig, axes = plt.subplots(n_rows, 1, figsize=(22, 13*n_rows))

for idx, fr_num in enumerate(frame_numbers):
    # TODO: Set vid to frame i
    # .set(): Method to set video properties
    # cv2.CAP_PROP_POS_FRAMES: Property flag indicating "frame position"
    # fr_num: The target frame number (e.g., 0, 10, 20, 30)
    vid.set(cv2.CAP_PROP_POS_FRAMES, fr_num)

    # TODO: Read a frame
    _, img = vid.read()
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # TODO: Get detections for frame i out of all_detections
    det = all_detections[fr_num]

    # TODO: Change det to the np.int32 dtype
    det = det.astype(np.int32)

    det_full = all_detections[fr_num].astype(np.int32)
    class_indices = det_full[:, 0]
    bboxes = det_full[:, 1:5]

# NEW_HEIGHT = 600
# scale = NEW_HEIGHT/park_img.shape[0]
# h, w = park_img_rescaled.shape[:2]
# detection_lbls_rescaled = (detection_lbls * (1, w, h, w, h)).astype(np.int32)

    # TODO: Draw on img

    draw_detections(img, bboxes, class_index=class_indices)
    annotate_class(img, bboxes, class_index=class_indices, class_map=COCO_CLASSES)


    axes[idx].imshow(img)
    axes[idx].axis('off')
    axes[idx].set_title(f'Frame #{fr_num}- {len(det)}objects')
plt.savefig('./detections.png', bbox_inches='tight')
plt.show()
print("saved to './detections.png' ")

<Figure size 640x480 with 0 Axes>

saved to './detections.png' 
